In [55]:
%useLatestDescriptors
%use dataframe
@file:DependsOn("com.github.doyaaaaaken:kotlin-csv-jvm:1.7.0")

import com.github.doyaaaaaken.kotlincsv.dsl.csvWriter
import io.github.oshai.kotlinlogging.KotlinLogging.logger
import kotlin.reflect.full.declaredMemberProperties
import java.nio.file.Paths
import java.util.Locale
import kotlin.io.path.Path

enum class Mode { FLAT, RANDOM }
enum class Algorithm(val shortName: String) {
    FROMBACK("tsprcs"),
    DISTANCE("tsprce"),
    SPARSITY("tsprcs"),
    OP("op"),
}
enum class Context { ELIMINATION, BUDGET, CLUSTERING }
enum class BudgetFactor(val string: String, val value: Int) {
    BUDGET_30("0.3", 30),
    BUDGET_50("0.5", 50),
    BUDGET_70("0.7", 70),
    BUDGET_100("1.0", 100)
}
enum class Parameter(val value: String) {
    CLUSTER_ELIMINATION_THRESHOLD("clustereliminationthreshold"),
    ALPHA("clustereliminationrevenueweight"),
    BETA("clustereliminationsparsityweight"),
    EPSILON("maxbudgetfactor"),
    ZETA("budgetweight")
}

enum class BudgetWeight(val value: String) {
    ELZEIN("el"),
    EQUAL("eq"),
    SPARSITY("cs"),
    DISTANCE("cd")
}

enum class BudgetMin(val value: String) {
    MIN("ml"),
    CENTER("c"),
    NONE("")
}

enum class UseMax(val value: String) {
    USEMAX("mx"),
    NOUSEMAX(""),
}

val percentageFraction = 1
val colsWithoutPercentages = "5.00"
val gradient = 0.1
val width = 1.9

val context = Context.BUDGET
val parameter = Parameter.ZETA
val mode = Mode.RANDOM
val algorithm = Algorithm.OP
val budgetFactor = BudgetFactor.BUDGET_100

val budgetWeight = BudgetWeight.SPARSITY
val budgetMin = BudgetMin.MIN
val useMax = UseMax.USEMAX


val fileName = when (parameter) {
    Parameter.ALPHA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.0_1.25_0.25.csv"
    Parameter.CLUSTER_ELIMINATION_THRESHOLD -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.3_0.7_0.1.csv"
    Parameter.BETA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmn_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_-0.25_1.25_0.25.csv"
    Parameter.EPSILON -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_1.0_7.0_1.0.csv"
    Parameter.ZETA -> "param_results_base_${mode.name.lowercase()}_5_${budgetFactor.value}_25_fbckmd_${algorithm.shortName}_50_${budgetWeight.value}${budgetMin.value}${useMax.value}_${parameter.value}_0.2_7.0_1.0.csv"
}

val relativePath = "/op-solver-strict/results/${context.name.lowercase()}/comparison/"
val navigationPath = Paths.get(System.getProperty("user.dir"), "../../../..").normalize().toAbsolutePath().toString()

val path = Paths.get(navigationPath, relativePath, fileName).toString()

var df = DataFrame.readCsv(path)
df = df.remove { df.columns()[1] }
df

name,0.20,0.40,0.60,0.80,1.00,2.00,3.00,4.00,5.00,6.00,7.00
eil101,3279.000000,3233.000000,3272.000000,3280.000000,3248.000000,3257.000000,3277.000000,3302.000000,3329.000000,3311.000000,3337.000000
gil262,7266.000000,7084.000000,7000.000000,7329.000000,7309.000000,7235.000000,7209.000000,6879.000000,7120.000000,7231.000000,7244.000000
pr299,8346.000000,8676.000000,7924.000000,8627.000000,8470.000000,8459.000000,8476.000000,8384.000000,8454.000000,8161.000000,8530.000000
lin318,9924.000000,10025.000000,9639.000000,10018.000000,9868.000000,9976.000000,9963.000000,10024.000000,9980.000000,9992.000000,9762.000000
rd400,11364.000000,11560.000000,11410.000000,11218.000000,10882.000000,11004.000000,11279.000000,10921.000000,11523.000000,10922.000000,10942.000000
d493,16492.000000,15706.000000,15158.000000,15856.000000,15626.000000,15798.000000,14620.000000,15710.000000,16053.000000,15646.000000,15909.000000
u574,16599.000000,16848.000000,16041.000000,16777.000000,16430.000000,16820.000000,16409.000000,16628.000000,16465.000000,16615.000000,16552.000000
u724,21071.000000,20987.000000,20515.000000,20571.000000,20661.000000,20470.000000,20796.000000,20843.000000,21151.000000,21014.000000,21069.000000
pcb1173,31785.000000,31808.000000,31227.000000,31467.000000,32126.000000,31477.000000,31823.000000,31815.000000,31627.000000,32070.000000,31235.000000
fl1400,50396.000000,49385.000000,36586.000000,49543.000000,46600.000000,45929.000000,49648.000000,47978.000000,51393.000000,48778.000000,47925.000000


In [56]:
val bestValues = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        0
    } else {
        (col[row] as Double).toInt()
    }
}.map { row ->
    row.rowMaxOf<Int>()
}

bestValues

[3337, 7329, 8676, 10025, 11560, 16492, 16848, 21151, 32126, 51393, 66127]

In [57]:

 //max(maxRevenueDif,0.0)

val rowMaxValues = bestValues.mapIndexed { index, resultMax -> max(resultMax, (df[colsWithoutPercentages][index] as Number).toInt()) }

rowMaxValues

[3337, 7329, 8676, 10025, 11560, 16492, 16848, 21151, 32126, 51393, 66127]

In [58]:
val rowMinValues = df.map { row ->
    row.rowMinOfOrNull<Double>()
}.map { row -> row!!.toInt()}
rowMinValues

[3233, 6879, 7924, 9639, 10882, 14620, 16041, 20470, 31227, 36586, 64890]

In [59]:

val revenueDif = rowMaxValues.mapIndexed { index, maxEntry ->
    val minEntry = df[index].rowMinOf<Double>()
    (maxEntry - minEntry!!).toDouble() / maxEntry.toDouble()
}

fun getSaturation(gradient: Double, maxValue: Int, value: Int): String {
    return min(max((100 - ((maxValue - value) / (maxValue * gradient) * 100)), 0.0), 100.0).toInt().toString()
}

fun calculatePercentage(refValue: Int, compValue: Int): Double {
    return ((compValue.toDouble() - refValue.toDouble()) / refValue.toDouble())
}

fun formatePercentage(value: Double): String {
    return "${String.format(Locale.US, "%+.${percentageFraction}f", value * 100)}\\%"
}

fun getPercentage(row: DataRow<*>, compValue: Int): String {
    val refValue = (df.get(colsWithoutPercentages)[row] as Double).toInt()
    val percentage = calculatePercentage(refValue, compValue)
    return "{\\tiny${formatePercentage(percentage)}}"
}

val footer = df.convert { all() }.perRowCol { row, col ->
    if (col.name() == colsWithoutPercentages || col[row] is String) {
        10000.0
    } else {
        val refValue = (df.get(colsWithoutPercentages)[row] as Double).toInt()
        calculatePercentage(refValue, (col[row] as Double).toInt())
    }
}.mean().values().mapIndexed { index, it ->
    if (index == 0) {
        "avg diff"
    } else if (it is Double && it > 100.0) {
        "-"
    } else if (it is Double) {
        formatePercentage(it)
    } else {
        "${it.toString()}\\%"
    }
}.toList()
footer

[avg diff, -0.0\%, -0.4\%, -5.1\%, -0.4\%, -1.8\%, -1.8\%, -1.5\%, -1.7\%, -, -1.3\%, -1.3\%]

In [60]:

val stringdf = df.convert { all() }.perRowCol { row, col ->
    if (col[row] is String) {
        col[row].toString().split("-").first()
    } else if (col.name() == colsWithoutPercentages) {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}"
        } else {
            "\\cellcolor{cyan!$saturation} $value"
        }
    } else {
        val value = (col[row] as Double).toInt()
        val maxValue = rowMaxValues[row.index()]
        val saturation = getSaturation(gradient, maxValue, value)
        val percentage = getPercentage(row, value)
        if (bestValues[row.index()] == value) {
            "\\cellcolor{cyan!$saturation} \\textbf{$value*}$percentage"
        } else {
            "\\cellcolor{cyan!$saturation} $value$percentage"
        }
    }
}
stringdf

name,0.20,0.40,0.60,0.80,1.00,2.00,3.00,4.00,5.00,6.00,7.00
eil101,\cellcolor{cyan!82} 3279{\tiny-1.5\%},\cellcolor{cyan!68} 3233{\tiny-2.9\%},\cellcolor{cyan!80} 3272{\tiny-1.7\%},\cellcolor{cyan!82} 3280{\tiny-1.5\%},\cellcolor{cyan!73} 3248{\tiny-2.4\%},\cellcolor{cyan!76} 3257{\tiny-2.2\%},\cellcolor{cyan!82} 3277{\tiny-1.6\%},\cellcolor{cyan!89} 3302{\tiny-0.8\%},\cellcolor{cyan!97} 3329,\cellcolor{cyan!92} 3311{\tiny-0.5\%},\cellcolor{cyan!100} \textbf{3337*}{\...
gil262,\cellcolor{cyan!91} 7266{\tiny+2.1\%},\cellcolor{cyan!66} 7084{\tiny-0.5\%},\cellcolor{cyan!55} 7000{\tiny-1.7\%},\cellcolor{cyan!100} \textbf{7329*}{\...,\cellcolor{cyan!97} 7309{\tiny+2.7\%},\cellcolor{cyan!87} 7235{\tiny+1.6\%},\cellcolor{cyan!83} 7209{\tiny+1.3\%},\cellcolor{cyan!38} 6879{\tiny-3.4\%},\cellcolor{cyan!71} 7120,\cellcolor{cyan!86} 7231{\tiny+1.6\%},\cellcolor{cyan!88} 7244{\tiny+1.7\%}
pr299,\cellcolor{cyan!61} 8346{\tiny-1.3\%},\cellcolor{cyan!100} \textbf{8676*}{\...,\cellcolor{cyan!13} 7924{\tiny-6.3\%},\cellcolor{cyan!94} 8627{\tiny+2.0\%},\cellcolor{cyan!76} 8470{\tiny+0.2\%},\cellcolor{cyan!74} 8459{\tiny+0.1\%},\cellcolor{cyan!76} 8476{\tiny+0.3\%},\cellcolor{cyan!66} 8384{\tiny-0.8\%},\cellcolor{cyan!74} 8454,\cellcolor{cyan!40} 8161{\tiny-3.5\%},\cellcolor{cyan!83} 8530{\tiny+0.9\%}
lin318,\cellcolor{cyan!89} 9924{\tiny-0.6\%},\cellcolor{cyan!100} \textbf{10025*}{...,\cellcolor{cyan!61} 9639{\tiny-3.4\%},\cellcolor{cyan!99} 10018{\tiny+0.4\%},\cellcolor{cyan!84} 9868{\tiny-1.1\%},\cellcolor{cyan!95} 9976{\tiny-0.0\%},\cellcolor{cyan!93} 9963{\tiny-0.2\%},\cellcolor{cyan!99} 10024{\tiny+0.4\%},\cellcolor{cyan!95} 9980,\cellcolor{cyan!96} 9992{\tiny+0.1\%},\cellcolor{cyan!73} 9762{\tiny-2.2\%}
rd400,\cellcolor{cyan!83} 11364{\tiny-1.4\%},\cellcolor{cyan!100} \textbf{11560*}{...,\cellcolor{cyan!87} 11410{\tiny-1.0\%},\cellcolor{cyan!70} 11218{\tiny-2.6\%},\cellcolor{cyan!41} 10882{\tiny-5.6\%},\cellcolor{cyan!51} 11004{\tiny-4.5\%},\cellcolor{cyan!75} 11279{\tiny-2.1\%},\cellcolor{cyan!44} 10921{\tiny-5.2\%},\cellcolor{cyan!96} 11523,\cellcolor{cyan!44} 10922{\tiny-5.2\%},\cellcolor{cyan!46} 10942{\tiny-5.0\%}
d493,\cellcolor{cyan!100} \textbf{16492*}{...,\cellcolor{cyan!52} 15706{\tiny-2.2\%},\cellcolor{cyan!19} 15158{\tiny-5.6\%},\cellcolor{cyan!61} 15856{\tiny-1.2\%},\cellcolor{cyan!47} 15626{\tiny-2.7\%},\cellcolor{cyan!57} 15798{\tiny-1.6\%},\cellcolor{cyan!0} 14620{\tiny-8.9\%},\cellcolor{cyan!52} 15710{\tiny-2.1\%},\cellcolor{cyan!73} 16053,\cellcolor{cyan!48} 15646{\tiny-2.5\%},\cellcolor{cyan!64} 15909{\tiny-0.9\%}
u574,\cellcolor{cyan!85} 16599{\tiny+0.8\%},\cellcolor{cyan!100} \textbf{16848*}{...,\cellcolor{cyan!52} 16041{\tiny-2.6\%},\cellcolor{cyan!95} 16777{\tiny+1.9\%},\cellcolor{cyan!75} 16430{\tiny-0.2\%},\cellcolor{cyan!98} 16820{\tiny+2.2\%},\cellcolor{cyan!73} 16409{\tiny-0.3\%},\cellcolor{cyan!86} 16628{\tiny+1.0\%},\cellcolor{cyan!77} 16465,\cellcolor{cyan!86} 16615{\tiny+0.9\%},\cellcolor{cyan!82} 16552{\tiny+0.5\%}
u724,\cellcolor{cyan!96} 21071{\tiny-0.4\%},\cellcolor{cyan!92} 20987{\tiny-0.8\%},\cellcolor{cyan!69} 20515{\tiny-3.0\%},\cellcolor{cyan!72} 20571{\tiny-2.7\%},\cellcolor{cyan!76} 20661{\tiny-2.3\%},\cellcolor{cyan!67} 20470{\tiny-3.2\%},\cellcolor{cyan!83} 20796{\tiny-1.7\%},\cellcolor{cyan!85} 20843{\tiny-1.5\%},\cellcolor{cyan!100} \textbf{21151*},\cellcolor{cyan!93} 21014{\tiny-0.6\%},\cellcolor{cyan!96} 21069{\tiny-0.4\%}
pcb1173,\cellcolor{cyan!89} 31785{\tiny+0.5\%},\cellcolor{cyan!90} 31808{\tiny+0.6\%},\cellcolor{cyan!72} 31227{\tiny-1.3\%},\cellcolor{cyan!79} 31467{\tiny-0.5\%},\cellcolor{cyan!100} \textbf{32126*}{...,\cellcolor{cyan!79} 31477{\tiny-0.5\%},\cellcolor{cyan!90} 31823{\tiny+0.6\%},\cellcolor{cyan!90} 31815{\tiny+0.6\%},\cellcolor{cyan!84} 31627,\cellcolor{cyan!98} 32070{\tiny+1.4\%},\cellcolor{cyan!72} 31235{\tiny-1.2\%}
fl1400,\cellcolor{cyan!80} 50396{\tiny-1.9\%},\cellcolor{cyan!60} 49385{\tiny-3.9\%},\cellcolor{cyan!0} 36586{\tiny-28.8\%},\cellcolor{cyan!64} 49543{\t

In [61]:
fun getLatexTable(formating: String, amountColumns: String, title: String, header: String, label: String, caption: String, body: String): String {
    return """
    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ $formating  }
                \hline
                \multicolumn{$amountColumns}{|c|}{$title} \\
                \hline
                    $header \\
                \hline
                    $body
                \hline
            \end{tabular}
        \end{adjustbox}
        \caption{$caption}
        \label{$label}
    \end{table}
         """
}

val shortAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "TSPrfb"
    Algorithm.DISTANCE -> "TSPrce"
    Algorithm.SPARSITY -> "TSPrcs"
    Algorithm.OP -> "OP"
}
val mediumAlgString = when (algorithm) {
    Algorithm.FROMBACK -> "cluster removal from back"
    Algorithm.DISTANCE -> "cluster removal based on distance"
    Algorithm.SPARSITY -> "cluster removal based on sparsity"
    Algorithm.OP -> "implicit cluster removal"
}

val amountColumns = df.columns().size.toString()
val formating = "|"+ df.columns().joinToString(separator = "") { "p{${width}cm}|" }
val header = df.columnNames().joinToString(separator = " & ")
val label = "tab:$shortAlgString:${mode.name.lowercase()}:${budgetFactor.value}"
val title = "Parameter search: \$R'\$ with \\textit{$shortAlgString}, ${mode.name.lowercase()}, \$ \\gamma = ${budgetFactor.string}\$."
//Parameter run for $R'$ using cluster removal from back with a budget of $\gamma = 0.5$. The percentage value refers to the mean revenue increase compared to $e^{blr}$. The highest revenue of an instance has 100\% saturation decreasing to 0\% at 70\% of the maximum. $\textbf{*}$ refers to the best mean revenue.
val caption = "Parameter run for \$R'\$ using $mediumAlgString with $\\gamma = ${budgetFactor.string}\$. The percentage value refers to the mean revenue increase compared to \$e^{bl${mode.name.lowercase().first().toString()}}$. The highest revenue of an instance has 100\\% saturation decreasing to 0\\% at ${(100 + gradient * -100).toInt()}\\% of the maximum revenue. The larges revenue value is referenced by \$\\textbf{*}\$."

val body = stringdf.rows().joinToString(separator = " \\\\ \n") { row ->
    row.values().joinToString(separator = " & ") {
        it.toString()
    }
} + " \\\\ \\hline " + footer.joinToString(separator = " & ") {
    it.toString()
} + " \\\\"

getLatexTable(formating, amountColumns, title, header, label, caption, body)


    \begin{table}[]
        \vspace{2em}
        \begin{adjustbox}{center}
            \begin{tabular}{ |p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|p{1.9cm}|  }
                \hline
                \multicolumn{12}{|c|}{Parameter search: $R'$ with \textit{OP}, random, $ \gamma = 1.0$.} \\
                \hline
                    name & 0.20 & 0.40 & 0.60 & 0.80 & 1.00 & 2.00 & 3.00 & 4.00 & 5.00 & 6.00 & 7.00 \\
                \hline
                    eil101 & \cellcolor{cyan!82} 3279{\tiny-1.5\%} & \cellcolor{cyan!68} 3233{\tiny-2.9\%} & \cellcolor{cyan!80} 3272{\tiny-1.7\%} & \cellcolor{cyan!82} 3280{\tiny-1.5\%} & \cellcolor{cyan!73} 3248{\tiny-2.4\%} & \cellcolor{cyan!76} 3257{\tiny-2.2\%} & \cellcolor{cyan!82} 3277{\tiny-1.6\%} & \cellcolor{cyan!89} 3302{\tiny-0.8\%} & \cellcolor{cyan!97} 3329 & \cellcolor{cyan!92} 3311{\tiny-0.5\%} & \cellcolor{cyan!100} \textbf{3337*}{\tiny+0.2\%} \\ 
gil262 & \cellcolor{cyan!91} 72